# Normalize corpus files

In [1]:
# import CONLL-U validator
#pip install regex
#import sys
#sys.path.append('/home/krzys/Programy/conll/ud-tools')
#import validate
import os
corpus_dir = os.path.expanduser('~/Kod/fontes_nlp/data/out/')
corpus_files = [ f for f in os.listdir(corpus_dir) if f.endswith("conllu") ]
corpus_dir_normalized = os.path.expanduser('./normalized')
#file_to_validate = os.path.join(corpus_dir,'AGZ4_TEI_final.conllu')
validate = "/home/krzys/Programy/conll/ud-tools/validate.py"

In [2]:
def normalize_tokens(tokenList):
    for i, token in enumerate(tokenList):
        interp = [',', ':', ';', '-', '.', '?', '!']
        #deprel
        if token["deprel"] == '_':
            if i == 0:
                token["deprel"] = 'root'
            else:
                if token["upos"] == 'PUNCT':
                    token["deprel"] = 'punct'
                else:
                    token["deprel"] = 'dep'
        #head
        if token.get("head") is None or token.get("head") == '_' or int(token.get("head")) < 1:
            if i == 0:
                token["head"] = 0
            else:
                token["head"] = 1
        #misc
        if 'O' in token["misc"].keys():
            del token["misc"]["O"]
        
        #SpaceAfter
        if i < len(tokenList) -1 and tokenList[i+1]["form"] in interp and token["misc"].get("SpaceAfter", '') != 'Yes':
            token["misc"]["SpaceAfter"] = "No"

In [3]:
[Line 39 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-value-upos-not-permitted] Value Abl of feature Case is not permitted with UPOS ADV in language [la].
[Line 39 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-value-upos-not-permitted] Value Masc of feature Gender is not permitted with UPOS ADV in language [la].
[Line 39 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-value-upos-not-permitted] Value Sing of feature Number is not permitted with UPOS ADV in language [la].
[Line 43 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-value-unknown] Value Con is not documented for feature Mood in language [la].
[Line 46 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-value-unknown] Value Con is not documented for feature Mood in language [la].
[Line 48 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-upos-not-permitted] Feature Degree is not permitted with UPOS CCONJ in language [la].
[Line 135 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-value-unknown] Value Con is not documented for feature Mood in language [la].
[Line 171 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-upos-not-permitted] Feature Degree is not permitted with UPOS SCONJ in language [la].
[Line 173 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-value-upos-not-permitted] Value Pos of feature Degree is not permitted with UPOS NOUN in language [la].
[Line 178 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-upos-not-permitted] Feature Degree is not permitted with UPOS NUM in language [la].
[Line 207 Sent AGZ4_TEI_final.conllu-sent2]: [L4 Morpho feature-value-upos-not-permitted] Value Pos of feature Degree is not permitted with UPOS NOUN in language [la].
[Line 270 Sent AGZ4_TEI_final.conllu-sent5]: [L4 Morpho feature-value-upos-not-permitted] Value Dat of feature Case is not permitted with UPOS ADV in language [la].
[Line 270 Sent AGZ4_TEI_final.conllu-sent5]: [L4 Morpho feature-value-upos-not-permitted] Value Masc of feature Gender is not permitted with UPOS ADV in language [la].
[Line 270 Sent AGZ4_TEI_final.conllu-sent5]: [L4 Morpho feature-value-upos-not-permitted] Value Plur of feature Number is not permitted with UPOS ADV in language [la].
[Line 374 Sent AGZ4_TEI_final.conllu-sent6]: [L4 Morpho feature-value-upos-not-permitted] Value Pos of feature Degree is not permitted with UPOS NOUN in language [la].
[Line 386 Sent AGZ4_TEI_final.conllu-sent7]: [L4 Morpho feature-value-upos-not-permitted] Value Pos of feature Degree is not permitted with UPOS NOUN in language [la].
[Line 388 Sent AGZ4_TEI_final.conllu-sent7]: [L4 Morpho feature-value-upos-not-permitted] Value Pos of feature Degree is not permitted with UPOS NOUN in language [la].
[Line 403 Sent AGZ4_TEI_final.conllu-sent7]: [L4 Morpho feature-value-unknown] Value Con is not documented for feature Mood in language [la].
...suppressing further errors regarding Morpho
[Line 38941 Sent AGZ4_TEI_final.conllu-sent390]: [L4 Format invalid-word-with-space] 'cccc ad' in column LEMMA is not on the list of exceptions allowed to contain whitespace (data/tokens_w_space.LANG files).
[Line 54380 Sent AGZ4_TEI_final.conllu-sent611]: [L4 Format invalid-word-with-space] 'cccc ad' in column LEMMA is not on the list of exceptions allowed to contain whitespace (data/tokens_w_space.LANG files).
[Line 72928 Sent AGZ4_TEI_final.conllu-sent863]: [L1 Format missing-empty-line] Missing empty line after the last sentence.


SyntaxError: invalid syntax. Perhaps you forgot a comma? (1216554755.py, line 1)

In [12]:
def get_text(tokenList):
    text = []
    interp = [',', ':', ';', '-', '.', '?', '!']
    for i,token in enumerate(tokenList):
        token_txt = ""
        if token["misc"].get("SpaceAfter") == "Yes" or (token["misc"].get("SpaceAfter") != "No" and i < len(tokenList) -1):
            token_txt = token["form"] + ' '
        else:
            token_txt = token["form"]
        text.append(token_txt)
    
    return ''.join(text)

In [70]:
from conllu import parse    
#corpus = dict.fromkeys(corpus_files)
for file in corpus_files:
    with open(os.path.join(corpus_dir,file), "r", encoding="utf-8") as f:
        print(f"Processing {file}")
        txt = f.read()
        sents = parse(txt)
        
        sents.metadata["newdoc_id"] = file
    
        sent_id_n = 1
        for sent in sents:
            sent.metadata["sent_id"] = sents.metadata["newdoc_id"] + '-sent' + str(sent_id_n)
            sent_id_n += 1
            
            normalize_tokens(sent)
            sent.metadata["text"] = get_text(sent)
        with open(os.path.join(corpus_dir_normalized, file), 'w') as f_out:
            f_out.writelines([sentence.serialize() for sentence in sents])
        with open(os.path.join(corpus_dir_normalized, file), 'r+') as f_out:
            content = f_out.read()
            content = content.rstrip('\n')
            f_out.seek(0)
            f_out.write(content)
            f_out.truncate()
        #normalized_content = content.replace('\n\n$', '\n')
        #with open(os.path.join(corpus_dir_normalized, file), 'w') as f_out:
        #    f_out.write(normalized_content)

Processing VAd_TEI_final.conllu
Processing VITELO_Persp1_TEI_final.conllu
Processing StSyn3_TEI_final.conllu
Processing HUSSOW.Hyac_TEI_final.conllu
Processing APozn1_TEI_final.conllu
Processing PomnLw2_TEI_final.conllu
Processing AGZ4_TEI_final.conllu
Processing CantMAe_TEI_final.conllu
Processing CatEpCr2_TEI_final.conllu
Processing MATTH.Rat_TEI_final.conllu
Processing JANKO_TEI_final.conllu
Processing AKapSąd1Pozn_TEI_final.conllu
Processing AnnSCr_TEI_final.conllu
Processing KsgŁawKr_TEI_final.conllu
Processing FRANCO_TEI_final.conllu
Processing KsgRachKr1_TEI_final.conllu
Processing KsgHenr3_TEI_final.conllu
Processing VKyng_B_TEI_final.conllu
Processing RoczHenryk_A_TEI_final.conllu
Processing StSyn1_2_TEI_final.conllu
Processing PomnLw1_TEI_final.conllu
Processing VSalom_TEI_final.conllu
Processing RoczMiech_TEI_final.conllu
Processing HERBORD_TEI_final.conllu
Processing THOM.Med_TEI_final.conllu
Processing KalCzerw_TEI_final.conllu
Processing KsgHenr6_TEI_final.conllu
Processi

In [72]:
# normalize CONLLU annotation
import os
import subprocess

rules = "/home/krzys/Programy/conll/conllueditor/rules_efontes.txt"
conllueditor = "/home/krzys/Programy/conll/conllueditor/conllueditor-2.22.4/bin/replace.sh"

#testing: for file in [ f for f in os.listdir(corpus_dir_normalized) if f.startswith("AGZ4")]:
for file in [ f for f in os.listdir(corpus_dir_normalized) if f.endswith(".conllu")]:
    print(f"Processing {file}")
    input_file = os.path.join(corpus_dir_normalized, file)
    
    # First, read the input file content
    with open(input_file, 'r') as file_to_read:
        input_content = file_to_read.read()

    # Run the external script on the file's content (in memory)
    arguments = [rules, input_file]
    
    # Redirect the output to subprocess.PIPE so we can capture it
    result = subprocess.run([conllueditor] + arguments, capture_output=True, text=True)
    
    # Check if the process ran successfully
    if result.returncode == 0:
        # Overwrite the input file with the result output (stdout)
        with open(input_file, 'w') as output_file:
            content = result.stdout
            # remove double \n's
            content = content.rstrip('\n')
            #f_out.seek(0)
            #f_out.write(content)
            #f_out.truncate()
            
            output_file.write(content)
    else:
        # Optionally, handle errors
        print(f"Error processing {file}: {result.stderr}")

Processing VAd_TEI_final.conllu
Processing VITELO_Persp1_TEI_final.conllu
Processing StSyn3_TEI_final.conllu
Processing HUSSOW.Hyac_TEI_final.conllu
Processing APozn1_TEI_final.conllu
Processing PomnLw2_TEI_final.conllu
Processing AGZ4_TEI_final.conllu
Processing CantMAe_TEI_final.conllu
Processing CatEpCr2_TEI_final.conllu
Processing MATTH.Rat_TEI_final.conllu
Processing JANKO_TEI_final.conllu
Processing AKapSąd1Pozn_TEI_final.conllu
Processing AnnSCr_TEI_final.conllu
Processing KsgŁawKr_TEI_final.conllu
Processing FRANCO_TEI_final.conllu
Processing KsgRachKr1_TEI_final.conllu
Processing KsgHenr3_TEI_final.conllu
Processing VKyng_B_TEI_final.conllu
Processing RoczHenryk_A_TEI_final.conllu
Processing StSyn1_2_TEI_final.conllu
Processing PomnLw1_TEI_final.conllu
Processing VSalom_TEI_final.conllu
Processing RoczMiech_TEI_final.conllu
Processing HERBORD_TEI_final.conllu
Processing THOM.Med_TEI_final.conllu
Processing KalCzerw_TEI_final.conllu
Processing KsgHenr6_TEI_final.conllu
Processi

In [ ]:
# https://quest.ms.mff.cuni.cz/udvalidator/cgi-bin/unidep/langspec/specify_feature.pl?lcode=la
import subprocess

# Define the arguments you want to pass to validate.py
arguments = ["--level", "6", "--lang", "la", 'normalized/' + 'AGZ4_TEI_final.conllu']

# Call validate.py with arguments
result = subprocess.run([validate] + arguments, capture_output=True, text=True)

# Print the output of validate.py
print("Standard Output:", result.stdout)
print("Standard Error:", result.stderr)